# FIFA World Cup Prediction using XGBoost

This notebook contains the complete pipeline for:
1. Downloading the international football results dataset.
2. Processing the match data and resolving team name changes.
3. Simulating a chronological Elo rating system.
4. Constructing features without data leakage.
5. Splitting data into Train, Validation (2018 World Cup), and Test (2022 World Cup) sets.
6. Training and evaluating an XGBoost model.
7. Exporting the model and pre-computed team Elo/form dictionaries for web deployment.

## Step 1: Libraries and Setup
First, we import the necessary packages.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt

## Step 2: Data Loading and Name Cleaning
We pull the raw match data directly from GitHub and clean team name discrepancies (e.g. West Germany -> Germany).

In [ ]:
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
df = pd.read_csv(url)
print(f"Total matches loaded: {len(df)}")

# Ensure chronological order
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by='date').reset_index(drop=True)

# Clean team names mapping
def clean_team_name(team):
    mapping = {
        'West Germany': 'Germany',
        'Czechoslovakia': 'Czech Republic',
        'Soviet Union': 'Russia',
        'Yugoslavia': 'Serbia',
        'German DR': 'Germany'
    }
    return mapping.get(team, team)

df['home_team'] = df['home_team'].apply(clean_team_name)
df['away_team'] = df['away_team'].apply(clean_team_name)
df.head()

## Step 3: Define Elo & Tournament Helper Functions
We classify tournament tiers to apply appropriate K-factors:
- **World Cup** (K=60)
- **Continental Cups** (K=50)
- **Qualifiers** (K=40)
- **Friendlies** (K=30)

In [ ]:
def get_tournament_type(tournament):
    tourn = tournament.lower()
    if 'fifa world cup' in tourn and 'qualification' not in tourn:
        return 3
    elif any(x in tourn for x in ['euro', 'copa', 'african cup of nations', 'asian cup', 'gold cup', 'nations cup', 'nations league']):
        if 'qualification' in tourn or 'qualifying' in tourn:
            return 1
        return 2
    elif 'qualification' in tourn or 'qualifying' in tourn or 'prep' in tourn:
        return 1
    elif 'friendly' in tourn:
        return 0
    else:
        return 0

def get_k_factor(tourn_type):
    if tourn_type == 3:
        return 60
    elif tourn_type == 2:
        return 50
    elif tourn_type == 1:
        return 40
    else:
        return 30

## Step 4: Chronological Elo Simulation and Feature Engineering
To avoid target leakage, all features for a match are computed using team history *prior* to that match. Only after features are computed do we update the team's historical records (Elo, last-match date, recent form outcomes, and goals scored/conceded).

In [ ]:
# Target encoding: 0 = Home Win, 1 = Draw, 2 = Away Win
df['target'] = 1
df.loc[df['home_score'] > df['away_score'], 'target'] = 0
df.loc[df['home_score'] < df['away_score'], 'target'] = 2

# State trackers
team_elos = {}
team_last_date = {}
team_results = {}
team_goals_scored = {}
team_goals_conceded = {}
h2h_history = {}

features_list = []

print("Processing match histories and features...")
for idx, row in df.iterrows():
    date = row['date']
    home_team = row['home_team']
    away_team = row['away_team']
    home_score = row['home_score']
    away_score = row['away_score']
    tournament = row['tournament']
    neutral = 1 if row['neutral'] else 0
    
    tourn_type = get_tournament_type(tournament)
    k_factor = get_k_factor(tourn_type)
    
    # Get team Elo ratings before match
    home_elo = team_elos.get(home_team, 1500.0)
    away_elo = team_elos.get(away_team, 1500.0)
    elo_diff = home_elo - away_elo
    
    # Get weighted form (last 5 matches: 0.1, 0.15, 0.2, 0.25, 0.3)
    def get_form(team):
        history = team_results.get(team, [])
        if not history:
            return 0.5
        padded = [0.5] * (5 - len(history)) + history[-5:]
        weights = [0.1, 0.15, 0.2, 0.25, 0.3]
        return sum(w * val for w, val in zip(weights, padded))
        
    home_form = get_form(home_team)
    away_form = get_form(away_team)
    
    # Get recent average goals scored/conceded
    def get_goals_avg(team, scored=True):
        goals = team_goals_scored.get(team, []) if scored else team_goals_conceded.get(team, [])
        if not goals:
            return 1.0
        recent = goals[-5:]
        return sum(recent) / len(recent)
        
    home_goals_scored_avg = get_goals_avg(home_team, scored=True)
    away_goals_scored_avg = get_goals_avg(away_team, scored=True)
    home_goals_conceded_avg = get_goals_avg(home_team, scored=False)
    away_goals_conceded_avg = get_goals_avg(away_team, scored=False)
    
    # Days since last match
    def get_days_since_last(team, current_date):
        last_date = team_last_date.get(team)
        if last_date is None:
            return 180.0
        return float((current_date - last_date).days)
        
    days_since_home_last = get_days_since_last(home_team, date)
    days_since_away_last = get_days_since_last(away_team, date)
    
    # Head to head win rate
    def get_h2h_win_rate(t_home, t_away):
        key = tuple(sorted([t_home, t_away]))
        history = h2h_history.get(key, [])
        if not history:
            return 0.5
        outcomes = []
        for t1, t2, val in history[-10:]:
            if t_home == t1:
                outcomes.append(val)
            else:
                outcomes.append(1.0 - val)
        return sum(outcomes) / len(outcomes)
        
    h2h = get_h2h_win_rate(home_team, away_team)
    
    # Append features dict
    features_list.append({
        'elo_diff': elo_diff,
        'home_form': home_form,
        'away_form': away_form,
        'home_goals_scored_avg': home_goals_scored_avg,
        'away_goals_scored_avg': away_goals_scored_avg,
        'home_goals_conceded_avg': home_goals_conceded_avg,
        'away_goals_conceded_avg': away_goals_conceded_avg,
        'tournament_type': tourn_type,
        'neutral_venue': neutral,
        'days_since_home_last': days_since_home_last,
        'days_since_away_last': days_since_away_last,
        'head_to_head': h2h
    })
    
    # Now update tracking states after match results are known
    expected_home = 1.0 / (1.0 + 10.0**((away_elo - home_elo) / 400.0))
    expected_away = 1.0 - expected_home
    
    if home_score > away_score:
        actual_home, actual_away = 1.0, 0.0
        outcome_home, outcome_away = 1.0, 0.0
    elif home_score == away_score:
        actual_home, actual_away = 0.5, 0.5
        outcome_home, outcome_away = 0.5, 0.5
    else:
        actual_home, actual_away = 0.0, 1.0
        outcome_home, outcome_away = 0.0, 1.0
        
    team_elos[home_team] = home_elo + k_factor * (actual_home - expected_home)
    team_elos[away_team] = away_elo + k_factor * (actual_away - expected_away)
    
    team_last_date[home_team] = date
    team_last_date[away_team] = date
    
    team_results.setdefault(home_team, []).append(outcome_home)
    team_results.setdefault(away_team, []).append(outcome_away)
    
    team_goals_scored.setdefault(home_team, []).append(home_score)
    team_goals_conceded.setdefault(home_team, []).append(away_score)
    team_goals_scored.setdefault(away_team, []).append(away_score)
    team_goals_conceded.setdefault(away_team, []).append(home_score)
    
    key = tuple(sorted([home_team, away_team]))
    h2h_history.setdefault(key, []).append((key[0], key[1], outcome_home if key[0] == home_team else outcome_away))

# Merge features back to df
features_df = pd.DataFrame(features_list)
df = pd.concat([df, features_df], axis=1)
feature_names = list(features_df.columns)
print("Feature engineering complete.")

## Step 5: Data Split
We split matches chronologically:
- **Train**: All matches before `2018-01-01`
- **Validation**: 2018 World Cup matches
- **Test**: 2022 World Cup matches

In [ ]:
train_mask = df['date'] < pd.to_datetime('2018-01-01')
val_mask = (df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2018)
test_mask = (df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2022)

X_train, y_train = df.loc[train_mask, feature_names], df.loc[train_mask, 'target']
X_val, y_val = df.loc[val_mask, feature_names], df.loc[val_mask, 'target']
X_test, y_test = df.loc[test_mask, feature_names], df.loc[test_mask, 'target']

print(f"Train matches: {len(X_train)}")
print(f"Validation matches: {len(X_val)}")
print(f"Test matches: {len(X_test)}")

## Step 6: Train XGBoost Classifier
We configure an `XGBClassifier` with early stopping using the validation set to avoid overfitting.

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    early_stopping_rounds=15
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

## Step 7: Model Evaluation
We evaluate the classification performance on Accuracy and Log Loss across splits.

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

y_prob_train = model.predict_proba(X_train)
y_prob_val = model.predict_proba(X_val)
y_prob_test = model.predict_proba(X_test)

print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f} | Train Log Loss: {log_loss(y_train, y_prob_train):.4f}")
print(f"Val Accuracy: {accuracy_score(y_val, y_pred_val):.4f} | Val Log Loss: {log_loss(y_val, y_prob_val):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f} | Test Log Loss: {log_loss(y_test, y_prob_test):.4f}")

## Step 8: Feature Importance Plotting

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
xgb.plot_importance(model, ax=ax, importance_type='weight')
plt.title("XGBoost Feature Importance (Weight)")
plt.tight_layout()
plt.show()

## Step 9: Save Model and State Metadata
We save the XGBoost model, Elo list, and team form/goals history dictionary using `pickle` so the Flask web application can make prediction queries on future fixtures instantly.

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Serialize datetime object keys to string so they don't break on reload
team_last_date_str = {k: v.strftime('%Y-%m-%d') for k, v in team_last_date.items()}

team_forms_data = {
    'last_match_dates': team_last_date_str,
    'results_history': team_results,
    'goals_scored_history': team_goals_scored,
    'goals_conceded_history': team_goals_conceded,
    'h2h_history': h2h_history
}

with open('team_elos.pkl', 'wb') as f:
    pickle.dump(team_elos, f)

with open('team_forms.pkl', 'wb') as f:
    pickle.dump(team_forms_data, f)

with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("All assets saved successfully!")